
# SENTRY-10 &nbsp;·&nbsp; Notebook A — train, evaluate, export

**Acoustic Safety / Anomaly Monitor** · *4-person team project (Team 10), Speech Processing and Multimodal AI, Week 8*

This is the **training** notebook. It fine-tunes the MIT Audio Spectrogram Transformer
(`MIT/ast-finetuned-audioset-10-10-0.4593`) on an 11-class safety subset of ESC-50, measures it honestly
(in-domain, unseen impulsive negatives, a degraded microphone channel, and an optional cross-source test),
builds the context layer and the 8-beat demo night, caches the LLM alert text and the spoken PA audio,
and finally **pushes the model and the demo artifacts to the Hugging Face Hub** (`HF_REPO`).

> **The interview demo is Notebook B (`B_live_demo.ipynb`)** — Run all there and a Gradio link appears
> in a few minutes. You only need this notebook to reproduce training or to republish the model.

**▶ Reproduce: Runtime → Run all (TRAIN = True)** — roughly **25–35 minutes on a T4 GPU**
(Runtime → Change runtime type → T4). Before you start, add a Colab secret **`HF_TOKEN`** (write access)
if you want the last step to publish to the Hub; optionally add **`GROQ_API_KEY`** so the alert text is
written by the LLM (otherwise a deterministic template is cached).

Notebook B loads what this notebook produced and starts in under a minute.

### What gets built

| Step | What happens | Roughly |
|---|---|---|
| 1 | Download ESC-50, carve out an 11-class safety subset | 4 min |
| 2 | Turn every clip into a log-mel spectrogram and cache it | 5 min |
| 3 | Load the AST backbone, bolt on a fresh 11-way head | 1 min |
| 4 | Fine-tune | 12–18 min |
| 5 | Evaluate on the held-out ESC-50 fold | 1 min |
| 6 | Evaluate on a **completely different recording source** — the honest test | 2 min |
| 7 | Tune the confidence gate, build the demo timeline, cache the LLM alerts | 4 min |
| 8 | Export everything Notebook B needs, then push it to the Hugging Face Hub | 2 min |

### The one thing to understand before you start

A safety monitor is not a loudness detector. Zone Z1 in our case study runs at 78–84 dBA all night —
a threshold alarm there would either fire constantly or never. The whole point of a trained classifier
is to tell **one loud thing from another loud thing**.


In [ ]:
# ── Settings ───────────────────────────────────────────────────
# This is the TRAINING notebook (the interview demo is Notebook B).
# TRAIN = True : full run, then push the model + demo artifacts to the Hugging Face Hub.
# TRAIN = False: still runs every step, but skips the Hub upload at the end.
TRAIN = True
HF_USER = "tantg101"          # your Hugging Face username
HF_REPO = f"{HF_USER}/sentry10-ast-esc50"

from IPython.core.magic import register_cell_magic
@register_cell_magic
def train_only(line, cell):
    """Cells marked %%train_only run only when TRAIN = True."""
    if TRAIN:
        get_ipython().run_cell(cell)
    else:
        print("⏭ Skipped (set TRAIN = True to run this cell).")



---
## Step 0 — check the machine

Runtime → Change runtime type → **T4 GPU**. If this cell says "no GPU", stop and fix it. Training on
CPU will take hours.

In [ ]:

import subprocess, sys
gpu = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if gpu.returncode != 0:
    print("NO GPU. Runtime -> Change runtime type -> T4 GPU, then Runtime -> Restart session.")
else:
    for line in gpu.stdout.split("\n"):
        if "MiB" in line or "Driver" in line:
            print(line.strip())
print("\npython", sys.version.split()[0])

In [ ]:

# One install cell. ~90 s. Restart is NOT required after this.
!pip install -q "transformers>=4.44" "datasets>=2.20" accelerate evaluate \
               scikit-learn librosa soundfile groq gtts 2>&1 | tail -2
import transformers, torch
print("transformers", transformers.__version__, "| torch", torch.__version__,
      "| cuda", torch.cuda.is_available())


---
## Step 1 — configuration

Everything tunable lives in this one cell so you never have to hunt through the notebook.

By default (`USE_DRIVE = False`) everything is written to `/content/SENTRY10/artifacts` on the Colab VM
and published to the Hugging Face Hub at the end. Set `USE_DRIVE = True` to keep a copy on your own
Google Drive instead.

`AUDIOSAMPLES_DIR` is **optional**: it is the course `AudioSamples` library (the one with
`Baby_Crying.mp3`, `Saw_electric.mp3` and friends), which is course material and is **not** distributed
with this repo and **never uploaded** to the Hub. If you have it, upload the folder to `/content/AudioSamples`
and Step 7c (cross-source evaluation) runs; if not, Step 7c skips itself and everything else runs.


In [ ]:

import os, json, random, numpy as np, torch

# ------------------------------------------------------------------ reproducibility
SEED = 20261113
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------------ where things live
USE_DRIVE = False   # True = also keep the artifacts on your own Google Drive
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/SENTRY10"
else:
    ROOT = "/content/SENTRY10"

ARTIFACTS = f"{ROOT}/artifacts"      # <- everything Notebook B reads is written here
CACHE     = "/content/cache"         # local scratch, not worth putting on Drive

# This notebook CREATES artifacts/ (and its subfolders) for you. It does NOT create
# AudioSamples/ -- that is the course sample library and you have to upload it yourself.
# It does not have to live under ROOT; we go and look for it.
import glob

def find_audiosamples(root):
    candidates = [
        f"{root}/AudioSamples",                              # ROOT/AudioSamples
        os.path.abspath(f"{root}/../AudioSamples"),           # one level up
        os.path.abspath(f"{root}/../../AudioSamples"),        # two levels up
        "/content/AudioSamples",                             # uploaded straight into Colab
        "/content/drive/MyDrive/AudioSamples",
    ]
    for c in candidates:
        if glob.glob(f"{c}/*.mp3"):
            return c
    # last resort: search Drive. Slow on a large Drive, but it beats failing at minute 20.
    for hit in sorted(glob.glob("/content/drive/MyDrive/**/AudioSamples", recursive=True)):
        if glob.glob(f"{hit}/*.mp3"):
            return hit
    return None

AUDIOSAMPLES_DIR = find_audiosamples(ROOT)          # <- or just hard-code the path here

for d in (ARTIFACTS, CACHE, f"{ARTIFACTS}/demo_audio", f"{ARTIFACTS}/figures"):
    os.makedirs(d, exist_ok=True)

# AudioSamples is OPTIONAL. Step 7c uses it if present and skips itself if not. The
# false-alarm rate (7a) and the channel-robustness figure (7b) need no uploads at all,
# so a missing folder costs you one evaluation, not the run.
HAVE_SAMPLES = False
if AUDIOSAMPLES_DIR:
    _mp3 = glob.glob(f"{AUDIOSAMPLES_DIR}/*.mp3")
    _need = ["Baby_Crying.mp3", "Saw_electric.mp3", "Gunshot.mp3", "VacuumCleaner.mp3"]
    _missing = [n for n in _need if not os.path.exists(f"{AUDIOSAMPLES_DIR}/{n}")]
    HAVE_SAMPLES = len(_mp3) >= 20 and not _missing
    print(f"AudioSamples     -> {AUDIOSAMPLES_DIR}   ({len(_mp3)} mp3s)")
    if _missing:
        print(f"  the expected files are not here ({_missing}) - this looks like the wrong")
        print("  folder, so Step 7c will skip. Set AUDIOSAMPLES_DIR by hand if you know better.")
    elif len(_mp3) < 35:
        print(f"  only {len(_mp3)} mp3s; the full course library has ~43.")
else:
    print("AudioSamples     -> NOT FOUND. Step 7c (cross-source) will skip.")
    print("  Everything else runs. If you want that evaluation, upload the folder anywhere")
    print("  to /content/AudioSamples and re-run this cell, or set AUDIOSAMPLES_DIR by hand above.")

# ------------------------------------------------------------------ the task
TARGET_CLASSES = ["glass_breaking", "siren", "clock_alarm", "crying_baby", "chainsaw",
                  "hand_saw", "car_horn", "engine", "vacuum_cleaner", "footsteps"]
BACKGROUND = "background_other"           # 11th class, sampled from everything else
CLASSES = TARGET_CLASSES + [BACKGROUND]
LABEL2ID = {c: i for i, c in enumerate(CLASSES)}
ID2LABEL = {i: c for c, i in LABEL2ID.items()}

TRAIN_FOLDS, TEST_FOLD = [1, 2, 3, 4], 5   # ESC-50's own 5-fold split. Never mix them.
BG_PER_FOLD = 80                           # background clips drawn per fold

# Six ESC-50 categories held out of training COMPLETELY -- not as a target class and not
# as background either. Every one is an impulsive or percussive broadband sound, which is
# the acoustic family that a glass-break detector confuses itself with. Because the model
# has never seen them in any form, the rate at which it reports them as an event is an
# honest false-alarm figure, and at 240 clips it has a usable sample size.
HOLDOUT_NEG_CATS = ["fireworks", "can_opening", "door_wood_knock",
                    "mouse_click", "clapping", "keyboard_typing"]

# ------------------------------------------------------------------ the model
MODEL_ID   = "MIT/ast-finetuned-audioset-10-10-0.4593"
SR         = 16000
EPOCHS     = 6
BATCH      = 8
GRAD_ACC   = 2                             # effective batch 16, fits a T4 comfortably
LR_BACKBONE = 3e-5                         # gentle: the backbone already knows about audio
LR_HEAD     = 3e-4                         # aggressive: the head is random at step 0
WARMUP_RATIO = 0.1
SPECAUG_FREQ_MASK = 24                     # mel bins masked, max
SPECAUG_TIME_MASK = 80                     # frames masked, max

# Confidence gate. Below this the prediction is discarded and reported as background.
# Set here so every evaluation uses the same value; Step 7d sweeps it and asks you to
# confirm or change this number, with a plot to justify whichever you pick.
TAU = 0.45

print(f"{len(CLASSES)} classes:", ", ".join(CLASSES))
print("artifacts ->", ARTIFACTS)


---
## Step 2 — get ESC-50 and carve out the safety subset

**ESC-50** is 2000 environmental sound clips: 50 categories × 40 clips, each exactly 5 seconds,
44.1 kHz. It ships with an official 5-fold split, and the folds are grouped so that clips from the same
original recording never straddle a fold boundary. Respecting those folds is the difference between an
honest number and a flattering one.

We keep 10 categories that map onto real safety events, and build an 11th `background_other` class by
sampling from the other 40 categories. That negative class matters more than people expect: without it
the model has to assign *every* sound to one of the ten alarm classes, so a bird, a cough and a chair
scrape all become "glass breaking".

In [ ]:

import urllib.request, zipfile, pandas as pd, shutil

ESC_DIR = "/content/ESC-50-master"
if not os.path.exists(ESC_DIR):
    url = "https://github.com/karolpiczak/ESC-50/archive/master.zip"
    print("downloading ESC-50 (~600 MB, 2-4 min) ...")
    urllib.request.urlretrieve(url, "/content/esc50.zip")
    with zipfile.ZipFile("/content/esc50.zip") as z:
        z.extractall("/content")
    os.remove("/content/esc50.zip")
print("ESC-50 ready at", ESC_DIR)

meta = pd.read_csv(f"{ESC_DIR}/meta/esc50.csv")
print(meta.shape, "rows |", meta.category.nunique(), "categories |",
      sorted(meta.fold.unique()), "folds")
meta.head(3)

In [ ]:

# ---- build the 11-class table -------------------------------------------------
rng = np.random.default_rng(SEED)

targets = meta[meta.category.isin(TARGET_CLASSES)].copy()
targets["label"] = targets.category

# the held-out negatives are excluded from the background pool, so they stay truly unseen
others = meta[~meta.category.isin(TARGET_CLASSES + HOLDOUT_NEG_CATS)].copy()
bg_rows = []
for fold in sorted(meta.fold.unique()):
    pool = others[others.fold == fold]
    # stratify across the 40 remaining categories so the negative class is genuinely varied
    n_other_cats = pool.category.nunique()          # 34 = 50 - 10 targets - 6 held-out negatives
    per_cat = max(1, BG_PER_FOLD // n_other_cats)
    # groupby().sample() rather than .apply(): the apply form is deprecated in pandas >= 2.2
    take = pool.groupby("category", group_keys=False).sample(
        n=per_cat, random_state=SEED, replace=False)
    if len(take) < BG_PER_FOLD:
        extra = pool.drop(take.index).sample(BG_PER_FOLD - len(take), random_state=SEED)
        take = pd.concat([take, extra])
    bg_rows.append(take.head(BG_PER_FOLD))
bg = pd.concat(bg_rows)
bg["label"] = BACKGROUND

unseen_neg = meta[meta.category.isin(HOLDOUT_NEG_CATS)].copy()
unseen_neg["label"] = BACKGROUND
unseen_neg["path"] = ESC_DIR + "/audio/" + unseen_neg.filename
print(f"held out of training entirely: {len(unseen_neg)} clips across "
      f"{unseen_neg.category.nunique()} impulsive categories "
      f"({', '.join(sorted(unseen_neg.category.unique()))})")

data = pd.concat([targets, bg], ignore_index=True)
data["label_id"] = data.label.map(LABEL2ID)
data["path"] = ESC_DIR + "/audio/" + data.filename
data["split"] = np.where(data.fold == TEST_FOLD, "test", "train")

print(data.groupby(["split", "label"]).size().unstack(fill_value=0).T)
print("\ntrain", (data["split"] == "train").sum(),
      "| test", (data["split"] == "test").sum())
data.to_csv(f"{ARTIFACTS}/dataset_manifest.csv", index=False)


### Look at the data before you train on it

Ten minutes here saves an hour of confused debugging later. Two things to check: (a) do the classes
look visually distinct as spectrograms, and (b) is `background_other` genuinely varied rather than
accidentally all birdsong.

In [ ]:

import librosa, librosa.display, matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 4, figsize=(16, 8))
for ax, cls in zip(axes.ravel(), CLASSES):
    row = (data[(data["label"] == cls) & (data["split"] == "train")]
           .sample(1, random_state=SEED).iloc[0])
    y, _ = librosa.load(row.path, sr=SR)
    S = librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=SR, n_mels=128), ref=np.max)
    librosa.display.specshow(S, sr=SR, x_axis="time", y_axis="mel", ax=ax, cmap="magma")
    ax.set_title(cls, fontsize=10, fontweight="bold")
    ax.set_xlabel(""); ax.set_ylabel("")
axes.ravel()[-1].axis("off")
fig.suptitle("One example per class, as the model will see it (log-mel, 128 bins)",
             fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig(f"{ARTIFACTS}/figures/class_examples.png", dpi=140, bbox_inches="tight")
plt.show()

bg_cats = sorted(data[data["label"] == BACKGROUND]["category"].unique())
print(f"background_other is drawn from {len(bg_cats)} categories:")
print("  " + ", ".join(bg_cats))


---
## Step 3 — features

AST does not look at a waveform. It looks at a **picture of a waveform**: a 128-bin log-mel
spectrogram, 1024 frames long, computed with a 25 ms window and a 10 ms hop. That picture is then cut
into 16×16 patches and fed to a vision transformer exactly as if it were an image. This is the single
most important idea to be able to explain in the demo — "we turned the sound into a photograph and
gave it to an image model".

We precompute and cache every spectrogram to disk. Feature extraction is CPU-bound, so doing it once
up front rather than every epoch cuts training time by more than half.

In [ ]:

from transformers import ASTFeatureExtractor
from tqdm.auto import tqdm
from IPython.display import Audio, display
import soundfile as sf

fe = ASTFeatureExtractor.from_pretrained(MODEL_ID)
print("mel bins:", fe.num_mel_bins, "| target frames:", fe.max_length,
      "| sampling rate:", fe.sampling_rate)

def clip_to_fbank(path):
    y, _ = librosa.load(path, sr=SR, mono=True)
    return fe(y, sampling_rate=SR, return_tensors="np")["input_values"][0]  # (1024, 128)

# ---- feature cache ------------------------------------------------------------
# Three rules learned the hard way, all of them enforced below:
#
#   1. NEVER put an array of Python strings in an .npz. numpy stores it as a pickled
#      object array, and np.load refuses to read pickles back by default
#      ("Object arrays cannot be loaded when allow_pickle=False"). So the boolean
#      IS_TEST mask goes in, not the "train"/"test" strings.
#   2. Version the filename. A cache written by an older version of this notebook is
#      worse than no cache, because it fails halfway through a 30-minute run.
#   3. Treat a cache miss as normal and a cache read failure as a cache miss. Never
#      let a corrupt scratch file stop the run.
CACHE_VERSION = 2
CACHE_NPZ = f"{CACHE}/fbanks_v{CACHE_VERSION}.npz"

def _extract_all():
    Xf = np.zeros((len(data), fe.max_length, fe.num_mel_bins), dtype=np.float32)
    for i, p in enumerate(tqdm(data["path"].tolist(), desc="extracting")):
        Xf[i] = clip_to_fbank(p)
    return Xf

X = Y = IS_TEST = None
if os.path.exists(CACHE_NPZ):
    try:
        with np.load(CACHE_NPZ) as z:               # allow_pickle stays False, deliberately
            assert int(z["version"]) == CACHE_VERSION
            assert int(z["n"]) == len(data)
            X = z["X"].astype(np.float32)          # stored as float16 to halve the file
            Y = z["Y"]
            IS_TEST = z["IS_TEST"].astype(bool)
        print(f"loaded cached features {X.shape} from {CACHE_NPZ}")
    except Exception as e:
        print(f"cache at {CACHE_NPZ} unusable ({type(e).__name__}: {e}) - re-extracting")
        X = None

if X is None:
    X = _extract_all()
    Y = data["label_id"].to_numpy()
    IS_TEST = (data["split"].to_numpy() == "test")
    np.savez_compressed(CACHE_NPZ, X=X.astype(np.float16), Y=Y,
                        IS_TEST=IS_TEST, version=CACHE_VERSION, n=len(data))
    print(f"cached {X.shape} -> {CACHE_NPZ} "
          f"({os.path.getsize(CACHE_NPZ)/1e6:.0f} MB on disk)")

assert X.shape[0] == len(Y) == len(IS_TEST) == len(data)
print(f"feature tensor: {X.shape} {X.dtype} | mean {X.mean():.3f} std {X.std():.3f}")
print(f"split: {(~IS_TEST).sum()} train / {IS_TEST.sum()} test")


### SpecAugment — the cheapest accuracy you will ever buy

640 training clips is not many. The standard fix in audio ML is **SpecAugment**: during training,
black out a random horizontal stripe (some mel bins) and a random vertical stripe (some time frames)
of the spectrogram. The model is forced to recognise a chainsaw from a *partial* picture of a chainsaw,
which is exactly the robustness we need when a real machine hall is roaring underneath.

It costs nothing, it runs on the fly, and it is applied to the training set only — never to
validation or test, which would be cheating.

In [ ]:

import torch
from torch.utils.data import Dataset

class FbankDataset(Dataset):
    def __init__(self, X, Y, train=False):
        self.X, self.Y, self.train = X, Y, train

    def __len__(self):
        return len(self.Y)

    def __getitem__(self, i):
        x = torch.from_numpy(self.X[i].copy())          # (frames, mels)
        if self.train:
            # frequency masking
            f = np.random.randint(0, SPECAUG_FREQ_MASK + 1)
            if f:
                f0 = np.random.randint(0, x.shape[1] - f)
                x[:, f0:f0 + f] = 0.0
            # time masking
            t = np.random.randint(0, SPECAUG_TIME_MASK + 1)
            if t:
                t0 = np.random.randint(0, x.shape[0] - t)
                x[t0:t0 + t, :] = 0.0
        return {"input_values": x, "labels": torch.tensor(self.Y[i], dtype=torch.long)}

tr_idx, te_idx = np.where(~IS_TEST)[0], np.where(IS_TEST)[0]
ds_train = FbankDataset(X[tr_idx], Y[tr_idx], train=True)
ds_test  = FbankDataset(X[te_idx], Y[te_idx], train=False)
print(len(ds_train), "train |", len(ds_test), "test")

# sanity: show one clip before and after augmentation
s = ds_train[0]["input_values"].numpy()
fig, ax = plt.subplots(1, 2, figsize=(11, 3))
ax[0].imshow(X[tr_idx][0].T, origin="lower", aspect="auto", cmap="magma"); ax[0].set_title("original")
ax[1].imshow(s.T, origin="lower", aspect="auto", cmap="magma"); ax[1].set_title("SpecAugment applied")
for a in ax: a.set_xlabel("frame"); a.set_ylabel("mel bin")
plt.tight_layout(); plt.show()


---
## Step 4 — the model

`MIT/ast-finetuned-audioset-10-10-0.4593` is an Audio Spectrogram Transformer that has already been
trained on **AudioSet**: 2 million YouTube clips across 527 sound categories. It already knows what
glass, sirens and engines sound like in a general sense. What it does not know is *our* 11-way
decision, or the acoustics of our site.

So we do two things:

1. **Throw away the 527-way head** and put a fresh randomly-initialised `768 → 11` linear layer in its
   place. `ignore_mismatched_sizes=True` is what lets us do this without an error.
2. **Use two learning rates.** The backbone gets `3e-5` — a nudge, because wrecking two million clips
   of pretraining would be foolish. The new head gets `3e-4`, ten times higher, because it starts from
   noise and needs to catch up fast. This trick is called *discriminative learning rates* and it is
   worth mentioning in the demo; it is the difference between a fine-tune and a reset.

In [ ]:

from transformers import ASTForAudioClassification

model = ASTForAudioClassification.from_pretrained(
    MODEL_ID,
    num_labels=len(CLASSES),
    label2id=LABEL2ID,
    id2label=ID2LABEL,
    ignore_mismatched_sizes=True,          # <- lets us swap the 527-way head for an 11-way one
)
model.config.problem_type = "single_label_classification"

total = sum(p.numel() for p in model.parameters())
head  = sum(p.numel() for p in model.classifier.parameters())
print(f"total parameters   : {total/1e6:8.2f} M")
print(f"new head           : {head/1e3:8.2f} K  ({100*head/total:.3f} % of the model)")
print(f"backbone (adapted) : {(total-head)/1e6:8.2f} M")
print("\nhead:", model.classifier)


### Baseline first: how much of this do you get *without* fine-tuning?

Before touching a single weight, we measure what the **frozen** backbone can already do. We pull the
pooled 768-dimensional embedding out of the un-fine-tuned AST for every clip and fit a plain logistic
regression on top — a *linear probe*. It costs about a minute and it is the only honest way to answer
the question an examiner will certainly ask: *did the fine-tuning actually do anything, or was the
pretrained model already good enough?*

Keep this number. Quoting "we reached X" is unremarkable. Quoting "a linear probe on the frozen
features reaches Y, and fine-tuning takes it to X" is evidence.

In [ ]:

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

@torch.no_grad()
def pooled_embeddings(idx, bs=16):
    model.eval()
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(dev)
    out = []
    for i in tqdm(range(0, len(idx), bs), desc="frozen features"):
        batch = torch.from_numpy(X[idx[i:i + bs]]).to(dev)
        h = model.audio_spectrogram_transformer(input_values=batch).pooler_output
        out.append(h.float().cpu().numpy())
    return np.concatenate(out)

Etr, Ete = pooled_embeddings(tr_idx), pooled_embeddings(te_idx)
# multinomial is the default and the multi_class kwarg is deprecated from sklearn 1.5
probe = LogisticRegression(max_iter=2000, C=1.0)
probe.fit(Etr, Y[tr_idx])
probe_acc = probe.score(Ete, Y[te_idx])
probe_f1  = f1_score(Y[te_idx], probe.predict(Ete), average="macro")

print(f"\nLINEAR PROBE on frozen AST features")
print(f"  embedding dim        : {Etr.shape[1]}")
print(f"  fold-5 accuracy      : {probe_acc:.3f}")
print(f"  fold-5 macro-F1      : {probe_f1:.3f}")
print("\nThis is the number fine-tuning has to beat. If it does not, the fine-tune is not")
print("earning its place and you should say so rather than hide it.")


---
## Step 5 — fine-tune

Six epochs over 640 clips. On a T4 this is 12–18 minutes. Watch the eval accuracy column: it should
climb fast for two epochs then flatten. If it is still bouncing around 30% at epoch 3, something is
wrong with the labels — stop and check `dataset_manifest.csv`.

In [ ]:

import evaluate
from sklearn.metrics import f1_score
from transformers import TrainingArguments, Trainer

acc_metric = evaluate.load("accuracy")

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {
        "accuracy": acc_metric.compute(predictions=preds, references=p.label_ids)["accuracy"],
        "macro_f1": f1_score(p.label_ids, preds, average="macro"),
    }

# discriminative learning rates: backbone slow, new head fast
head_params = [p for n, p in model.named_parameters() if n.startswith("classifier")]
body_params = [p for n, p in model.named_parameters() if not n.startswith("classifier")]
optimizer = torch.optim.AdamW(
    [{"params": body_params, "lr": LR_BACKBONE},
     {"params": head_params, "lr": LR_HEAD}], weight_decay=0.01)

# transformers 5 removed warmup_ratio, so convert the ratio to a step count (works on 4.x and 5.x).
import math
steps_per_epoch = math.ceil(len(ds_train) / (BATCH * GRAD_ACC))
WARMUP_STEPS = max(1, int(steps_per_epoch * EPOCHS * WARMUP_RATIO))

args = TrainingArguments(
    output_dir=f"{CACHE}/train",
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACC,
    num_train_epochs=EPOCHS,
    warmup_steps=WARMUP_STEPS,
    lr_scheduler_type="cosine",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=10,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
)

trainer = Trainer(model=model, args=args, train_dataset=ds_train, eval_dataset=ds_test,
                  compute_metrics=compute_metrics, optimizers=(optimizer, None))
train_out = trainer.train()
print("\ntrain runtime: %.1f min" % (train_out.metrics["train_runtime"] / 60))

In [ ]:

# training curve -- goes in the report
hist = trainer.state.log_history
tr = [(h["epoch"], h["loss"]) for h in hist if "loss" in h and "eval_loss" not in h]
ev = [(h["epoch"], h["eval_loss"], h["eval_accuracy"], h["eval_macro_f1"])
      for h in hist if "eval_accuracy" in h]

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(*zip(*tr), label="train loss", lw=1, alpha=.7)
ax[0].plot([e[0] for e in ev], [e[1] for e in ev], "o-", label="eval loss", lw=2)
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss"); ax[0].legend(); ax[0].set_title("loss")
ax[1].plot([e[0] for e in ev], [e[2] for e in ev], "o-", label="accuracy", lw=2)
ax[1].plot([e[0] for e in ev], [e[3] for e in ev], "s-", label="macro F1", lw=2)
ax[1].set_xlabel("epoch"); ax[1].set_ylim(0, 1); ax[1].legend(); ax[1].set_title("fold-5 performance")
for a in ax: a.grid(alpha=.25)
fig.tight_layout(); fig.savefig(f"{ARTIFACTS}/figures/training_curve.png", dpi=140,
                                bbox_inches="tight")
plt.show()
for e in ev:
    print(f"epoch {e[0]:.0f}  eval_loss {e[1]:.3f}  acc {e[2]:.3f}  macroF1 {e[3]:.3f}")


---
## Step 6 — in-domain evaluation (ESC-50 fold 5)

This is the number that will look good. It is measured on clips the model has never seen, but from the
same corpus, recorded by the same community, mastered the same way. Report it, but do not stop here.

In [ ]:

from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

pred_out = trainer.predict(ds_test)
logits = pred_out.predictions
probs  = torch.softmax(torch.tensor(logits), dim=1).numpy()
y_pred = probs.argmax(1)
y_true = Y[te_idx]

in_domain_acc = (y_pred == y_true).mean()
in_domain_f1  = f1_score(y_true, y_pred, average="macro")

# Recall on the ten target classes only, ignoring background. This is the figure that is
# comparable with the recall numbers from Steps 7a-7c, so it is computed here and reused.
_det = y_true != LABEL2ID[BACKGROUND]
in_domain_recall = float((y_pred[_det] == y_true[_det]).mean())

print(f"fine-tuned      accuracy {in_domain_acc:.3f}   macro-F1 {in_domain_f1:.3f}   "
      f"target recall {in_domain_recall:.3f}")
print(f"linear probe    accuracy {probe_acc:.3f}   macro-F1 {probe_f1:.3f}   (frozen backbone)")
print(f"WHAT THE FINE-TUNE BOUGHT  {in_domain_acc - probe_acc:+.3f} accuracy, "
      f"{in_domain_f1 - probe_f1:+.3f} macro-F1")
print("\nThat delta is the answer to \"did you actually fine-tune anything?\". Memorise it.\n")
print(classification_report(y_true, y_pred, target_names=CLASSES, digits=3, zero_division=0))

In [ ]:

def plot_cm(y_true, y_pred, classes, title, path, normalise=True):
    cm = confusion_matrix(y_true, y_pred, labels=range(len(classes)))
    disp = cm.astype(float)
    if normalise:
        disp = disp / np.clip(cm.sum(1, keepdims=True), 1, None)
    fig, ax = plt.subplots(figsize=(8.6, 7.2))
    im = ax.imshow(disp, cmap="Blues", vmin=0, vmax=1 if normalise else disp.max())
    for i in range(len(classes)):
        for j in range(len(classes)):
            if cm[i, j]:
                ax.text(j, i, f"{cm[i, j]}", ha="center", va="center", fontsize=8,
                        color="white" if disp[i, j] > 0.5 else "#1c2430",
                        fontweight="bold" if i == j else "normal")
    ax.set_xticks(range(len(classes))); ax.set_xticklabels(classes, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(classes))); ax.set_yticklabels(classes, fontsize=8)
    ax.set_xlabel("predicted"); ax.set_ylabel("true")
    ax.set_title(title, fontsize=12, fontweight="bold", pad=12)
    fig.colorbar(im, ax=ax, shrink=.8)
    fig.tight_layout(); fig.savefig(path, dpi=140, bbox_inches="tight"); plt.show()
    return cm

cm_in = plot_cm(y_true, y_pred, CLASSES,
                f"In-domain: ESC-50 fold 5  (acc {in_domain_acc:.3f})",
                f"{ARTIFACTS}/figures/confusion_in_domain.png")


---
## Step 7a — false alarms on sounds the model has never seen

Before we go anywhere near an external sample library, here is a false-alarm test that needs no
uploads at all and has a real sample size.

Six ESC-50 categories were held out of training **completely** — not as a target class, and not
inside `background_other` either. Fireworks, a can opening, a wooden door knock, a mouse click,
clapping, keyboard typing. Every one is an impulsive broadband transient: acoustically the same
family as breaking glass, which is the confusion a detector like this actually suffers from.

240 clips the model has never seen in any form. Every one of them should come back as
`background_other`. The proportion that does not is our false-alarm rate, and unlike a hand-picked
set of eight it is a number with a defensible denominator.

In [ ]:

@torch.no_grad()
def predict_paths(paths, chunk_s=5.0, hop_s=2.5, max_win=8, desc="predicting"):
    # 5 s windows, 2.5 s hop, keep the window the model is most confident about --
    # the same rule the live monitor uses.
    model.eval()
    rows = []
    for p in tqdm(paths, desc=desc):
        y, _ = librosa.load(p, sr=SR, mono=True)
        n, hop = int(chunk_s * SR), int(hop_s * SR)
        if len(y) < n:
            y = np.pad(y, (0, n - len(y)))
        wins = [y[i:i + n] for i in range(0, max(1, len(y) - n + 1), hop)][:max_win]
        feats = fe(wins, sampling_rate=SR, return_tensors="pt")["input_values"].to(model.device)
        pr = torch.softmax(model(input_values=feats).logits.float(), dim=1).cpu().numpy()
        best = pr.max(1).argmax()
        rows.append({"file": os.path.basename(p), "probs": pr[best],
                     "pred": ID2LABEL[int(pr[best].argmax())], "conf": float(pr[best].max())})
    return pd.DataFrame(rows)

un = predict_paths(unseen_neg.path.tolist(), desc="unseen negatives")
un["category"] = unseen_neg.category.to_numpy()
un["false_alarm"] = un.pred != BACKGROUND

unseen_far = un.false_alarm.mean()
print(f"\nUNSEEN IMPULSIVE NEGATIVES  ({len(un)} clips, never seen in training)")
print(f"  false-alarm rate : {unseen_far:.3f}   ({un.false_alarm.sum()} of {len(un)})")
print()
print(un.groupby("category").agg(clips=("file", "size"),
                                 false_alarms=("false_alarm", "sum"),
                                 rate=("false_alarm", "mean")).to_string())
print()
worst = un[un.false_alarm].pred.value_counts()
if len(worst):
    print("what they get called instead of background:")
    print(worst.to_string())
    print("\nIf glass_breaking dominates that list, say so in the demo before anyone asks.")
else:
    print("no false alarms at all on 240 unseen impulsive clips - state the n when you quote this.")


## Step 7b — a simulated change of recording chain

The other half of the honesty problem is not *which sounds* but *which microphone*. ESC-50 fold 5 is
held out, but it was recorded by the same community with the same equipment as folds 1–4.

So we take the fold-5 clips and push them through a deliberately unflattering channel: a band-limit
that mimics a cheap MEMS capsule, a small room reverb, added noise at 10 dB SNR, a level change, and
— if ffmpeg is available — a 64 kbps MP3 round trip. Then we re-measure recall.

**Be precise about what this proves.** It measures robustness to a degraded channel. It is *not* the
same as testing on genuinely different source recordings, because the underlying performance is
identical. It is a lower bound on the domain-shift problem, not a measurement of it. Say it that way.

In [ ]:

import shutil, subprocess, tempfile
from scipy.signal import butter, sosfilt, fftconvolve

def cheap_mic_chain(y, sr=SR, snr_db=10.0, rng=None):
    rng = rng or np.random.default_rng(SEED)
    # 1. band-limit: 180 Hz - 6.5 kHz, roughly a cheap capsule in a plastic housing
    sos = butter(4, [180 / (sr / 2), 6500 / (sr / 2)], btype="band", output="sos")
    y = sosfilt(sos, y).astype(np.float32)
    # 2. small room: exponentially decaying noise impulse response, RT60 ~ 0.35 s
    ir_n = int(0.35 * sr)
    ir = (rng.normal(0, 1, ir_n) * np.exp(-np.linspace(0, 8, ir_n))).astype(np.float32)
    ir[0] = 1.0
    y = fftconvolve(y, ir / np.abs(ir).sum(), mode="same").astype(np.float32)
    # 3. noise at the requested SNR
    sig_p = np.mean(y ** 2) + 1e-12
    y = y + rng.normal(0, np.sqrt(sig_p / (10 ** (snr_db / 10))), len(y)).astype(np.float32)
    # 4. a level the recordist did not think about
    y = y * rng.uniform(0.35, 0.9)
    return np.clip(y, -1.0, 1.0)

HAVE_FFMPEG = shutil.which("ffmpeg") is not None
def mp3_roundtrip(y, sr=SR, kbps=64):
    if not HAVE_FFMPEG:
        return y
    with tempfile.TemporaryDirectory() as d:
        sf.write(f"{d}/a.wav", y, sr)
        subprocess.run(["ffmpeg", "-y", "-loglevel", "quiet", "-i", f"{d}/a.wav",
                        "-b:a", f"{kbps}k", f"{d}/a.mp3"], check=True)
        out, _ = librosa.load(f"{d}/a.mp3", sr=sr, mono=True)
    return out.astype(np.float32)

print("ffmpeg available for the MP3 leg:", HAVE_FFMPEG)

@torch.no_grad()
def eval_degraded(rows, snr_db=10.0):
    model.eval()
    rng = np.random.default_rng(SEED)
    preds = []
    for p in tqdm(rows["path"].tolist(), desc=f"channel shift @ {snr_db:.0f} dB SNR"):
        y, _ = librosa.load(p, sr=SR, mono=True)
        y = mp3_roundtrip(cheap_mic_chain(y, snr_db=snr_db, rng=rng))
        f_ = fe(y, sampling_rate=SR, return_tensors="pt")["input_values"].to(model.device)
        pr = torch.softmax(model(input_values=f_).logits.float(), dim=1)[0].cpu().numpy()
        preds.append(int(pr.argmax()) if pr.max() >= TAU else LABEL2ID[BACKGROUND])
    return np.array(preds)

held_rows = data[data["split"] == "test"].reset_index(drop=True)
held_y = held_rows["label_id"].to_numpy()
deg_pred = eval_degraded(held_rows)

is_target = held_y != LABEL2ID[BACKGROUND]
clean_recall = in_domain_recall
deg_recall = float((deg_pred[is_target] == held_y[is_target]).mean())
deg_acc = float((deg_pred == held_y).mean())

print(f"\nfold 5, clean channel      : recall {clean_recall:.3f} on the 10 target classes")
print(f"fold 5, degraded channel   : recall {deg_recall:.3f}   accuracy {deg_acc:.3f}")
print(f"COST OF THE CHANNEL        : {deg_recall - clean_recall:+.3f} recall")
print("\nThis is robustness to a bad microphone, NOT robustness to a different source.")
print("Do not present it as a cross-source result. Name it 'simulated channel shift'.")

# listen to what we did -- worth 10 seconds of the demo if anyone asks
_ex = held_rows.iloc[0]
_y, _ = librosa.load(_ex["path"], sr=SR, mono=True)
print(f"\nexample: {_ex['label']}  ({_ex['filename']})")
display(Audio(_y, rate=SR))
display(Audio(mp3_roundtrip(cheap_mic_chain(_y)), rate=SR))


---
## Step 7c — cross-source evaluation on the course library  *(optional)*

This is the strongest evidence available, and it is the one part of the notebook that needs a file
you have to upload. **If `AudioSamples` is not present this section skips itself** and the run
continues — Steps 7a and 7b already gave you a false-alarm rate and a channel-robustness figure.

Now the interesting part. We take the **course `AudioSamples` library** — 42 MP3s recorded on different
equipment, in different rooms, in a different decade, and MP3-compressed on top. None of it was ever in
ESC-50. If the model only learned "what ESC-50 microphones sound like", this is where it falls apart.

We split them deliberately:

* **Positives** — clips that genuinely belong to one of our classes (`Baby_Crying` → `crying_baby`,
  `Saw_electric` → `chainsaw`, and so on). Some of these are *near* matches rather than exact, and we
  flag which. There are only a handful of exact positives, so we report the count next to the number
  every single time. A recall of 0.80 on n=5 is a sentence, not a result.
* **Hard negatives** — ten clips chosen because they are the classic confusions for an impulsive
  detector: a gunshot, firecrackers, a champagne cork, tearing cloth, a stapler, a hammer, a cartoon
  boing, a door knock and two typewriters. All broadband transients. All should come back as
  `background_other`. Any of them coming back as `glass_breaking` is a false alarm we need to know
  about *before* the demo, not during it.

**Do not report a single accuracy over this whole set.** Most of it is negatives, so an
always-`background_other` classifier would look good. The cell below splits it into recall on
positives and specificity on negatives, and only compares recall to recall when it quotes a
generalisation gap.

In [ ]:

import glob

# filename -> (expected class, is_exact_match)
CROSS_SOURCE = {
    # ---------------- positives
    "Baby_Crying.mp3":               ("crying_baby",     True),
    "Kittens_Crying.mp3":            ("crying_baby",     False),  # cat, not infant -- deliberately hard
    "Saw_electric.mp3":              ("chainsaw",        True),
    "Razor_Electric.mp3":            ("chainsaw",        False),  # small motor, not a saw
    "VacuumCleaner.mp3":             ("vacuum_cleaner",  True),
    "Blender.mp3":                   ("vacuum_cleaner",  False),
    "Footsteps_Clogs.mp3":           ("footsteps",       True),
    "Traffic_inCity.mp3":            ("engine",          True),
    "Aeroplane_Jet_Landing.mp3":     ("engine",          False),
    "Aeroplane_Propeller_Flying.mp3":("engine",          False),
    "Car_Trye_Screech.mp3":          ("car_horn",        False),
    "Telephone_Dial_Ring_Hangup.mp3":("clock_alarm",     False),
    "Buzzer_Door_Open.mp3":          ("clock_alarm",     False),
    "Clock_Cuckoo.mp3":              ("clock_alarm",     False),
    "Kettle.mp3":                    ("clock_alarm",     False),
    # ---------------- hard negatives (see HARD_NEGATIVES below): SHOULD be background_other
    "Gunshot.mp3":                   ("background_other", True),
    "Fireworks_Firecrackers.mp3":    ("background_other", True),
    "Bottle_Champaigne.mp3":         ("background_other", True),
    "Tear_Cloth.mp3":                ("background_other", True),
    "StaplerGun.mp3":                ("background_other", True),
    "Hammer.mp3":                    ("background_other", True),
    "Boing.mp3":                     ("background_other", True),
    "Typewriter.mp3":                ("background_other", True),
    "Typewriter_2.mp3":              ("background_other", True),
    "Doorknock_open.mp3":            ("background_other", True),
    # ---------------- easy negatives: nature and crowd
    "Birds_inPark.mp3":              ("background_other", True),
    "Birds_Forest_Brook.mp3":        ("background_other", True),
    "Birds_Sparrows_Blackbirds.mp3": ("background_other", True),
    "Seagulls.mp3":                  ("background_other", True),
    "Geese.mp3":                     ("background_other", True),
    "Chickens.mp3":                  ("background_other", True),
    "Cat.mp3":                       ("background_other", True),
    "Cat_2.mp3":                     ("background_other", True),
    "Dog_Growl.mp3":                 ("background_other", True),
    "Lion.mp3":                      ("background_other", True),
    "Horse_Whinney.mp3":             ("background_other", True),
    "Cattledrive.mp3":               ("background_other", True),
    "InsectCricket.mp3":             ("background_other", True),
    "Applause_hall.mp3":             ("background_other", True),
    "Children_Laughing.mp3":         ("background_other", True),
    "Crowd_Angry.mp3":               ("background_other", True),
    "Carnival.mp3":                  ("background_other", True),
}
# Impulsive broadband transients: the nearest acoustic neighbours of breaking glass.
# All of these MUST come back as background_other. Kept as one list so the count in the
# report can never drift from the count in the code.
HARD_NEGATIVES = ["Gunshot.mp3", "Fireworks_Firecrackers.mp3", "Bottle_Champaigne.mp3",
                  "Tear_Cloth.mp3", "StaplerGun.mp3", "Hammer.mp3", "Boing.mp3",
                  "Doorknock_open.mp3", "Typewriter.mp3", "Typewriter_2.mp3"]
assert all(CROSS_SOURCE[k][0] == "background_other" for k in HARD_NEGATIVES)

if not HAVE_SAMPLES:
    print("SKIPPING Step 7c - no usable AudioSamples folder.")
    print("Steps 7a and 7b already gave you a false-alarm rate and a channel-shift figure.")
    xs = None
available = ({os.path.basename(p): p for p in glob.glob(f"{AUDIOSAMPLES_DIR}/*.mp3")}
             if HAVE_SAMPLES else {})
missing = [k for k in CROSS_SOURCE if k not in available]
usable = [k for k in CROSS_SOURCE if k in available]
if HAVE_SAMPLES:
    print(f"{len(available)} mp3s in {AUDIOSAMPLES_DIR}")
    print(f"{len(usable)} of our {len(CROSS_SOURCE)} labelled files are present")
    if missing:
        print("MISSING (skipped):", missing)

In [ ]:

xs = xs_all = xs_pos = xs_pos_exact = xs_neg = None
pos = pos_exact = neg = None

if HAVE_SAMPLES:
    xs_paths = [available[k] for k in CROSS_SOURCE if k in available]
    xs = predict_paths(xs_paths, max_win=24, desc="cross-source")   # defined in Step 7a
    xs["pred_id"] = xs.pred.map(LABEL2ID)
    xs["true"] = xs.file.map(lambda f: CROSS_SOURCE[f][0])
    xs["exact"] = xs.file.map(lambda f: CROSS_SOURCE[f][1])
    xs["true_id"] = xs.true.map(LABEL2ID)
    xs["correct"] = xs.pred == xs.true
    xs = xs.sort_values(["true", "file"]).reset_index(drop=True)

    # Reporting note. An overall accuracy on this set is close to meaningless, because most of
    # the clips are negatives -- a model that answered "background_other" to everything would
    # score well. So we split it three ways and the report quotes all three.
    xs["is_pos"] = xs.true != BACKGROUND
    pos       = xs[xs.is_pos]
    pos_exact = xs[xs.is_pos & xs.exact]
    neg       = xs[~xs.is_pos]

    xs_all       = xs.correct.mean()
    xs_pos       = pos.correct.mean()
    xs_pos_exact = pos_exact.correct.mean()
    xs_neg       = neg.correct.mean()

    print(f"clips evaluated                             : {len(xs)}"
          f"  ({len(pos)} positive, {len(neg)} negative)")
    print(f"RECALL on positives, exact mappings only    : {xs_pos_exact:.3f}   (n={len(pos_exact)})")
    print(f"RECALL on positives, incl. approximate      : {xs_pos:.3f}   (n={len(pos)})")
    print(f"SPECIFICITY on negatives                    : {xs_neg:.3f}   (n={len(neg)})")
    print(f"overall accuracy on the whole set           : {xs_all:.3f}"
          f"   <- dominated by negatives, do not headline this")
    print()
    print(f"in-domain accuracy (ESC-50 fold 5, balanced): {in_domain_acc:.3f}")
    print(f"in-domain RECALL on the same 10 target classes: {in_domain_recall:.3f}")
    print(f"GENERALISATION GAP (recall vs recall)       : "
          f"{in_domain_recall - xs_pos_exact:+.3f}   <- like for like")
    print()
    print("n=%d is a small sample. Quote it every time you quote the number." % len(pos_exact))
    xs[["file", "true", "pred", "conf", "exact", "correct"]].to_string(index=False)
else:
    print("Cross-source evaluation skipped (no AudioSamples folder).")
    print(f"Use these instead, both already computed:")
    print(f"  false alarms on 240 unseen impulsive clips : {unseen_far:.3f}")
    print(f"  recall through a degraded channel          : {deg_recall:.3f} "
          f"(vs {clean_recall:.3f} clean)")

In [ ]:

# the false-alarm question the audience will ask
hn = fa = None
if HAVE_SAMPLES:
    hn = xs[xs.file.isin(HARD_NEGATIVES)]
    fa = hn[hn.pred != BACKGROUND]
    print("HARD NEGATIVES -- broadband transients that are not breaking glass\n")
    for _, r in hn.iterrows():
        flag = "  <-- FALSE ALARM" if r.pred != BACKGROUND else ""
        print(f"  {r.file:<32} -> {r.pred:<17} {r.conf:.2f}{flag}")
    print(f"\nfalse-alarm rate on hard negatives: {len(fa)}/{len(hn)} = "
          f"{len(fa)/max(1,len(hn)):.2f}")
    print(f"compare with {unseen_far:.2f} on the 240 unseen ESC-50 impulsive clips (Step 7a) - "
          f"the AudioSamples set is smaller but genuinely off-corpus.")

    cm_xs = plot_cm(xs.true_id.to_numpy(), xs.pred_id.to_numpy(), CLASSES,
                    f"Cross-source: course AudioSamples\n"
                    f"recall {xs_pos_exact:.2f} on {len(pos_exact)} exact positives   |   "
                    f"specificity {xs_neg:.2f} on {len(neg)} negatives",
                    f"{ARTIFACTS}/figures/confusion_cross_source.png")
    xs.drop(columns=["probs"]).to_csv(f"{ARTIFACTS}/cross_source_results.csv", index=False)


### Choosing the confidence gate

The model always returns *something*. A gate `τ` says: below this confidence, call it
`background_other` and stay quiet. Raising `τ` cuts false alarms and also cuts real detections. There
is no free lunch, and the sweep below shows you exactly what the trade costs. Pick a value, write it in
the report, and be ready to justify it.

For a safety monitor the asymmetry matters: a missed glass-break is worse than an extra alert, so we
lean low. But an operator who gets twenty false alarms a night stops reading them, which turns a low
gate into a *different* kind of missed detection. That tension is worth one sentence in the demo.

In [ ]:

taus = np.arange(0.20, 0.96, 0.05)
# The sweep uses the 240 UNSEEN impulsive clips, not the hand-picked eight: always
# available, and a denominator you can defend.
hn_conf = np.stack(un.probs.values)                                # hoisted out of the loop
det = (y_true != LABEL2ID[BACKGROUND])
rows = []
for t in taus:
    gated = np.where(probs.max(1) >= t, probs.argmax(1), LABEL2ID[BACKGROUND])
    recall = (gated[det] == y_true[det]).mean()
    hn_pred = np.where(hn_conf.max(1) >= t, hn_conf.argmax(1), LABEL2ID[BACKGROUND])
    far = (hn_pred != LABEL2ID[BACKGROUND]).mean()
    rows.append((t, recall, far))
sweep = pd.DataFrame(rows, columns=["tau", "recall_on_real_events", "false_alarm_rate"])

fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.plot(sweep.tau, sweep.recall_on_real_events, "o-", label="recall on real events", lw=2)
ax.plot(sweep.tau, sweep.false_alarm_rate, "s-", lw=2, color="#c0392b",
        label=f"false alarms on {len(un)} unseen impulsive clips")
ax.set_xlabel("confidence gate  tau"); ax.set_ylabel("rate"); ax.set_ylim(-.02, 1.02)
ax.grid(alpha=.25); ax.legend(); ax.set_title("What the gate costs you", fontweight="bold")
fig.tight_layout(); fig.savefig(f"{ARTIFACTS}/figures/tau_sweep.png", dpi=140, bbox_inches="tight")
plt.show()
print(sweep.to_string(index=False))

# Confirm or change the value set in Step 1, now that you can see what it costs.
TAU = 0.45      # <-- justify this number in the report
print(f"\nconfirmed tau = {TAU}")
print("Everything above this cell used the Step 1 value. If you change tau here, re-run")
print("Steps 7a-7c so the reported figures match the gate you are actually shipping.")


---
## Step 8 — build the demo assets

Two things happen here.

1. **Rebuild the incident timeline from held-out fold-5 clips.** The `PREVIEW` wav shipped in the repo
   was assembled from the course sample library plus three synthesised sources, so that you can
   rehearse without a GPU. For the real demo we want clips the model has genuinely never trained on,
   which is exactly what fold 5 is. Same script, better provenance.
2. **Cache everything.** Predictions, severities and LLM alert texts get written to
   `demo_cache.json`. Notebook B can then run in `SAFE` mode — real model, cached text — or `REPLAY`
   mode with no network and no GPU at all. This is your insurance policy for demo day.

In [ ]:

import soundfile as sf, json

SPEC = {
  "zones": {
    "Z1": {"name": "Machine Hall",              "type": "Industrial"},
    "Z2": {"name": "Chemical & Solvent Store",  "type": "Hazmat"},
    "Z3": {"name": "Nightshift Childcare Annexe","type": "Welfare"},
    "Z4": {"name": "Perimeter Yard",            "type": "Outdoor"},
  },
  "timeline": [
    {"t":   0.0, "clock": "22:04", "zone": "Z1", "truth": "engine",         "beat": "BASELINE"},
    {"t":  22.0, "clock": "22:31", "zone": "Z3", "truth": "crying_baby",    "beat": "SUPPRESS"},
    {"t":  44.0, "clock": "23:47", "zone": "Z4", "truth": "car_horn",       "beat": "BASELINE"},
    {"t":  66.0, "clock": "01:12", "zone": "Z1", "truth": "chainsaw",       "beat": "FIRST_ALERT"},
    {"t":  92.0, "clock": "01:14", "zone": "Z1", "truth": "glass_breaking", "beat": "ESCALATE"},
    {"t": 118.0, "clock": "01:16", "zone": "Z2", "truth": "clock_alarm",    "beat": "CRITICAL"},
    {"t": 146.0, "clock": "01:17", "zone": "Z3", "truth": "crying_baby",    "beat": "PUNCHLINE"},
    {"t": 174.0, "clock": "01:22", "zone": "Z4", "truth": "siren",          "beat": "RESOLVE"},
  ],
}

# Pick one held-out fold-5 clip per event -- the model has never seen any of these.
#
# CRITICAL: a repeated (class, zone) pair must reuse the SAME file. The 22:31 and 01:17
# crying_baby events are the punchline of the whole demo, and the punchline is that the
# audio is byte-for-byte identical while the severity is not. If these two events get
# different clips the demo still runs and quietly stops making its point.
held = data[data["split"] == "test"]
chosen, used, by_key = {}, set(), {}
for ev in SPEC["timeline"]:
    key = (ev["truth"], ev["zone"])
    if key in by_key:
        row = by_key[key]
        ev["repeat_of"] = row.filename
    else:
        pool = held[(held.label == ev["truth"]) & (~held.filename.isin(used))]
        if len(pool) == 0:
            pool = held[held.label == ev["truth"]]
        row = pool.sample(1, random_state=SEED + int(ev["t"])).iloc[0]
        used.add(row.filename)
        by_key[key] = row
    chosen[ev["t"]] = row.path
    ev["source_clip"] = row.filename

for i, ev in enumerate(SPEC["timeline"]):
    y, _ = librosa.load(chosen[ev["t"]], sr=SR, mono=True)
    y = y / (np.abs(y).max() + 1e-9) * 0.85
    name = f"ev_{i:02d}_{ev['zone']}_{ev['truth']}.wav"
    sf.write(f"{ARTIFACTS}/demo_audio/{name}", y, SR, subtype="PCM_16")
    ev["file"] = name
    ev["index"] = i

reps = [(e["clock"], e["source_clip"]) for e in SPEC["timeline"] if "repeat_of" in e]
print("wrote", len(SPEC["timeline"]), "held-out event clips to", f"{ARTIFACTS}/demo_audio")
for clock, src in reps:
    print(f"  {clock} deliberately reuses {src}  <- this is the punchline; do not 'fix' it")
assert len(reps) >= 1, "the punchline event is missing - check the timeline"


### The context layer

This is the part that turns a classifier into a system, and it is where most of the "creative
integration" marks live. Three rules, then an LLM.

* **Zone-conditional severity.** Breaking glass in the machine hall is probably a dropped bottle.
  Breaking glass at the perimeter fence at 01:00 is probably a break-in. Same class, same confidence,
  different consequence.
* **Out-of-hours modifier.** Between 21:00 and 06:00, cutting tools and footsteps get one step more
  serious, because nothing is scheduled.
* **Rolling correlation.** Events inside a five-minute window are related until proven otherwise. Three
  of them across two zones is not three alerts, it is one incident.

The LLM then receives *only the event list* — never audio, never a transcript — and writes the alert a
human actually reads.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  THE CONTEXT LAYER
#  This is the part that turns a sound classifier into a safety system, and it is
#  the part worth talking about in the demo. Four rules, in this order:
#
#    1. ZONE      the same class means different things in different rooms
#    2. HOURS     out-of-hours activity is more suspicious than the same thing at noon
#    3. EXCEPTIONS  two named carve-outs that pure arithmetic gets wrong
#    4. CORRELATION  events inside a 5-minute window are one situation, not several
#
#  Every rule is hand-written and auditable. That is deliberate: an escalation
#  policy is a safety document. A safety officer has to be able to read it,
#  disagree with a line, and change that line. A learned policy cannot offer that.
# ─────────────────────────────────────────────────────────────────────────────
LADDER = ["INFO", "LOW", "MEDIUM", "HIGH", "CRITICAL"]

BASE_SEV = {"glass_breaking": "HIGH",  "siren": "MEDIUM",   "clock_alarm": "HIGH",
            "crying_baby":    "MEDIUM","chainsaw": "MEDIUM","hand_saw": "LOW",
            "car_horn":       "LOW",   "engine": "INFO",    "vacuum_cleaner": "INFO",
            "footsteps":      "INFO",  "background_other": "INFO"}

# steps up (+) or down (-) the ladder, per zone
ZONE_MOD = {
 "Z1": {"glass_breaking": -1, "crying_baby": +2, "chainsaw": 0,  "hand_saw": 0,
        "engine": -1, "vacuum_cleaner": -1, "footsteps": -1},
 "Z2": {"glass_breaking": +2, "crying_baby": +2, "chainsaw": +2, "hand_saw": +2,
        "engine": +2, "vacuum_cleaner": +1, "footsteps": +2, "clock_alarm": +1},
 "Z3": {"glass_breaking": +1, "crying_baby": -2, "chainsaw": +2, "hand_saw": +2,
        "engine": +1, "footsteps": -1, "clock_alarm": +1},
 "Z4": {"glass_breaking": +1, "crying_baby": +2, "car_horn": -1, "engine": -1,
        "footsteps": 0},
}

OOH_CLASSES      = {"glass_breaking", "chainsaw", "hand_saw", "footsteps", "engine"}
RESPONSE_CLASSES = {"siren"}       # emergency response ARRIVING is not a new threat
DISTRESS_CLASSES = {"crying_baby"} # human distress: suppression is revoked mid-incident
WINDOW_MIN       = 5


def _mins(clock):
    h, m = int(clock[:2]), int(clock[3:])
    return (h if h >= 12 else h + 24) * 60 + m      # night shift wraps past midnight


def in_window(a, b):
    return abs(_mins(a) - _mins(b)) <= WINDOW_MIN


def severity(cls, zone, clock, recent, return_raw=False):
    # recent = list of dicts {clock, zone, label, severity} already inside the window
    #
    # return_raw=True additionally returns the UNCAPPED ladder position. The cap exists
    # because CRITICAL means "site-wide incident" and a single event must never declare
    # one -- but that means several zones can land on the same capped answer for very
    # different reasons, and a panel that compares zones needs to see through it.
    step, why = LADDER.index(BASE_SEV.get(cls, "INFO")), []
    incident_open = any(r["severity"] == "CRITICAL" for r in recent)

    # --- rule 1 + exception A: zone, unless suppression has been revoked ----------
    zmod = ZONE_MOD.get(zone, {}).get(cls, 0)
    if zmod < 0 and incident_open and cls in DISTRESS_CLASSES:
        why.append(f"zone {zone} {zmod:+d} SUPPRESSION REVOKED (incident open)")
        zmod = 0
    if zmod:
        step += zmod
        why.append(f"zone {zone} {zmod:+d}")

    # --- rule 2: out of hours ------------------------------------------------------
    hour = int(clock[:2])
    if (hour >= 21 or hour < 6) and cls in OOH_CLASSES:
        step += 1
        why.append("out of hours +1")

    # --- exception B: response classes never escalate -------------------------------
    if cls in RESPONSE_CLASSES:
        why.append("response class, exempt from escalation")
        out = LADDER[max(0, min(4, step))], why
        return (*out, step) if return_raw else out

    # --- rule 4: correlation --------------------------------------------------------
    prior = [r for r in recent if LADDER.index(r["severity"]) >= LADDER.index("MEDIUM")]
    cross = (not incident_open and len(prior) >= 2
             and len({r["zone"] for r in prior} | {zone}) >= 2)
    if prior and LADDER.index(BASE_SEV.get(cls, "INFO")) >= LADDER.index("LOW"):
        step += 1
        why.append(f"correlated with {len(prior)} recent alert(s) +1")
    if cross:
        step += 1
        why.append("cross-zone cascade +1")

    # Only a cross-zone cascade may declare CRITICAL. Once an incident is already
    # open we stop ratcheting: further events are scored on their own merits.
    cap = 4 if cross else 3
    out = LADDER[max(0, min(cap, step))], why
    return (*out, step) if return_raw else out

# ---- does the script actually play out the way we designed it? -------------------
_recent = []
print(f"{'clock':<7}{'zone':<5}{'class':<16}{'severity':<10} reasoning")
for _ev in SPEC["timeline"]:
    _sev, _why = severity(_ev["truth"], _ev["zone"], _ev["clock"], _recent)
    print(f"{_ev['clock']:<7}{_ev['zone']:<5}{_ev['truth']:<16}{_sev:<10}{'; '.join(_why)}")
    _recent = [r for r in _recent if in_window(r["clock"], _ev["clock"])]
    _recent.append({"clock": _ev["clock"], "zone": _ev["zone"],
                    "label": _ev["truth"], "severity": _sev})

# The two beats the whole demo hangs on:
print("\nsame class, same zone, different night:")
print("  crying_baby Z3 22:31, quiet night   ->", severity("crying_baby","Z3","22:31",[])[0])
print("  crying_baby Z3 01:17, incident open ->",
      severity("crying_baby","Z3","01:17",
               [{"clock":"01:16","zone":"Z2","label":"clock_alarm","severity":"CRITICAL"},
                {"clock":"01:14","zone":"Z1","label":"glass_breaking","severity":"HIGH"}])[0])

In [ ]:

# ---- run the model over the demo clips and record what it actually says ----------
recent, cache_events = [], []
for ev in SPEC["timeline"]:
    p = f"{ARTIFACTS}/demo_audio/{ev['file']}"
    y, _ = librosa.load(p, sr=SR, mono=True)
    with torch.no_grad():
        f_ = fe(y, sampling_rate=SR, return_tensors="pt")["input_values"].to(model.device)
        pr = torch.softmax(model(input_values=f_).logits, dim=1)[0].cpu().numpy()
    pred = ID2LABEL[int(pr.argmax())] if pr.max() >= TAU else BACKGROUND
    conf = float(pr.max())
    sev, why = severity(pred, ev["zone"], ev["clock"], recent)
    top3 = [(ID2LABEL[i], round(float(pr[i]), 3)) for i in pr.argsort()[::-1][:3]]
    rec = {**ev, "pred": pred, "conf": round(conf, 3), "severity": sev,
           "reasons": why, "top3": top3, "model_correct": pred == ev["truth"]}
    cache_events.append(rec)
    recent = [r for r in recent if in_window(r["clock"], ev["clock"])]
    recent.append({"clock": ev["clock"], "zone": ev["zone"], "label": pred, "severity": sev})
    mark = "OK " if rec["model_correct"] else "MISS"
    print(f"{mark} {ev['clock']} {ev['zone']} truth={ev['truth']:<15} pred={pred:<15} "
          f"{conf:.2f}  -> {sev:<8} {'; '.join(why)}")


### Cache the LLM alerts

We call Groq once, here, and store the result. On demo day Notebook B can either call it live (looks
better, small risk) or read this cache (no risk at all). Either way the words in the room are the same
words we tested, which is the whole point of caching.

Put your key in Colab's **Secrets** panel (the key icon in the left sidebar) as `GROQ_API_KEY` and
enable notebook access. If it is missing, the deterministic template fallback runs instead and the
notebook carries on — nothing here blocks on the network.

In [ ]:

from google.colab import userdata

SYSTEM_PROMPT = (
 "You are the alert writer for SENTRY-10, an acoustic safety monitor at a mixed-use industrial site "
 "in Singapore. You receive ONLY a list of detected sound events with zone, time, confidence and a "
 "computed severity. You never receive audio and never receive speech transcripts.\n\n"
 "Write the alert a night-shift supervisor should read. Rules:\n"
 "1. Two sentences maximum. Lead with what to DO, not with what was heard.\n"
 "2. Name the zone in plain language, not the code.\n"
 "3. If several events are correlated, describe them as ONE situation, not a list.\n"
 "4. State the confidence honestly. If it is below 0.7, say the detection is uncertain.\n"
 "5. Never invent detail that is not in the event data. No casualties, no causes, no speculation "
 "about people unless a human-distress class was actually detected.\n"
 "6. Plain English. No emoji, no markdown, no preamble."
)

def fallback_alert(ev, recent):
    z = SPEC["zones"][ev["zone"]]["name"]
    if ev["severity"] in ("INFO", "LOW"):
        return f"No action required. {ev['pred'].replace('_',' ').capitalize()} detected in the {z} at {ev['clock']}; consistent with normal operation."
    lead = {"MEDIUM": "Check", "HIGH": "Attend", "CRITICAL": "Evacuate and attend"}[ev["severity"]]
    extra = f" This follows {len(recent)} related detection(s) in the last five minutes." if recent else ""
    unsure = " Detection confidence is low; verify before escalating." if ev["conf"] < 0.7 else ""
    return (f"{lead} the {z} now: {ev['pred'].replace('_',' ')} detected at {ev['clock']} "
            f"(confidence {ev['conf']:.2f}).{extra}{unsure}")

def call_groq(ev, recent):
    from groq import Groq
    client = Groq(api_key=userdata.get("GROQ_API_KEY"))
    payload = {
        "current_event": {"time": ev["clock"], "zone": ev["zone"],
                          "zone_name": SPEC["zones"][ev["zone"]]["name"],
                          "zone_type": SPEC["zones"][ev["zone"]]["type"],
                          "sound": ev["pred"], "confidence": ev["conf"],
                          "computed_severity": ev["severity"],
                          "why_this_severity": ev["reasons"]},
        "recent_events_last_5_min": recent,
    }
    r = client.chat.completions.create(
        model="llama-3.3-70b-versatile", temperature=0.3, max_tokens=140,
        messages=[{"role": "system", "content": SYSTEM_PROMPT},
                  {"role": "user", "content": json.dumps(payload)}])
    return r.choices[0].message.content.strip()

have_key = False
try:
    have_key = bool(userdata.get("GROQ_API_KEY"))
except Exception:
    pass
print("GROQ_API_KEY present:", have_key)

recent = []
for ev in cache_events:
    rc = [{"time": r["clock"], "zone": r["zone"], "sound": r["label"],
           "severity": r["severity"]} for r in recent]
    try:
        ev["alert"] = call_groq(ev, rc) if have_key else fallback_alert(ev, rc)
        ev["alert_source"] = "groq" if have_key else "template"
    except Exception as e:
        ev["alert"] = fallback_alert(ev, rc); ev["alert_source"] = f"template (groq failed: {e})"
    recent = [r for r in recent if in_window(r["clock"], ev["clock"])]
    recent.append({"clock": ev["clock"], "zone": ev["zone"],
                   "label": ev["pred"], "severity": ev["severity"]})
    print(f"\n[{ev['severity']}] {ev['clock']} {ev['zone']}  ({ev['alert_source']})\n  {ev['alert']}")

In [ ]:

# ---- pre-render the PA announcements so demo day needs no network for audio ------
from gtts import gTTS

os.makedirs(f"{ARTIFACTS}/demo_audio/pa", exist_ok=True)
for ev in cache_events:
    if ev["severity"] in ("INFO", "LOW"):
        continue
    chime = {"MEDIUM": "Attention.", "HIGH": "Attention. Priority alert.",
             "CRITICAL": "Attention. Emergency. Emergency."}[ev["severity"]]
    text = f"{chime} {ev['alert']}"
    out = f"{ARTIFACTS}/demo_audio/pa/pa_{int(ev['t']):03d}.mp3"
    try:
        gTTS(text, lang="en", tld="com.au").save(out)     # Lab B: tld changes the accent
        ev["pa_file"] = os.path.basename(out)
        print("rendered", os.path.basename(out), f"({len(text)} chars)")
    except Exception as e:
        print("gTTS failed for", ev["clock"], e)

In [ ]:

# ---- write the single file Notebook B depends on --------------------------------
demo_cache = {
    "built_at": pd.Timestamp.now().isoformat(timespec="seconds"),
    "model_id": MODEL_ID, "tau": TAU, "classes": CLASSES,
    "root": ROOT, "audiosamples_dir": AUDIOSAMPLES_DIR,
    "metrics": {
        "in_domain_accuracy": round(float(in_domain_acc), 4),
        "in_domain_macro_f1": round(float(in_domain_f1), 4),
        "in_domain_recall_targets": round(float(in_domain_recall), 4),
        "linear_probe_accuracy": round(float(probe_acc), 4),
        "unseen_negative_false_alarm_rate": round(float(unseen_far), 4),
        "n_unseen_negatives": int(len(un)),
        "channel_shift_recall": round(float(deg_recall), 4),
        "channel_shift_accuracy": round(float(deg_acc), 4),
        "have_cross_source": bool(HAVE_SAMPLES),
        "cross_source_recall_exact": (round(float(xs_pos_exact), 4) if HAVE_SAMPLES else None),
        "cross_source_recall_all_pos": (round(float(xs_pos), 4) if HAVE_SAMPLES else None),
        "cross_source_specificity": (round(float(xs_neg), 4) if HAVE_SAMPLES else None),
        "cross_source_accuracy_all": (round(float(xs_all), 4) if HAVE_SAMPLES else None),
        "n_cross_source": (int(len(xs)) if HAVE_SAMPLES else 0),
        "n_cross_source_pos_exact": (int(len(pos_exact)) if HAVE_SAMPLES else 0),
        "n_cross_source_pos": (int(len(pos)) if HAVE_SAMPLES else 0),
        "n_hard_negatives": (int(len(hn)) if HAVE_SAMPLES else 0),
        "hard_negative_false_alarm_rate": (round(float(len(fa) / max(1, len(hn))), 4)
                                           if HAVE_SAMPLES else None),
        "n_train": int(len(ds_train)), "n_test": int(len(ds_test)),
        "epochs": EPOCHS, "train_minutes": round(train_out.metrics["train_runtime"] / 60, 1),
    },
    "zones": SPEC["zones"],
    "events": cache_events,
    "tau_sweep": sweep.to_dict(orient="records"),
}
with open(f"{ARTIFACTS}/demo_cache.json", "w") as f:
    json.dump(demo_cache, f, indent=2, default=str)

trainer.save_model(f"{ARTIFACTS}/model")
fe.save_pretrained(f"{ARTIFACTS}/model")
print("saved model + feature extractor ->", f"{ARTIFACTS}/model")
print(json.dumps(demo_cache["metrics"], indent=2))

---
## Step 8b — publish to the Hugging Face Hub

Uploads the fine-tuned model and the demo artifacts to `HF_REPO`, which is where Notebook B downloads them
from. Runs only with `TRAIN = True` and a Colab secret `HF_TOKEN` (write access); otherwise it prints how
to enable it and the notebook carries on. The course `AudioSamples` library is never uploaded.


In [ ]:
%%train_only
# ── Publish the model + demo artifacts to the Hugging Face Hub ─────────────────
# Notebook B (the interview demo) downloads exactly this repo with snapshot_download().
# Needs a Colab secret HF_TOKEN with WRITE access (Secrets panel, key icon, notebook access on).
#
#   repo root         model.safetensors, config.json (with id2label), preprocessor_config.json
#   artifacts/        demo_cache.json (local paths removed), demo_audio/ev_0*.wav (8 ESC-50
#                     fold-5 clips), demo_audio/pa/pa_*.mp3 (pre-rendered PA), figures/
#
# The course AudioSamples library is course material: it is NEVER uploaded. Only the
# 8 ESC-50 demo clips (CC BY-NC) and files this notebook generated go up.
import json, os, shutil
from huggingface_hub import HfApi

hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception as e:
        print("Could not read the HF_TOKEN secret:", repr(e))

if not hf_token:
    print("No HF_TOKEN secret found - skipping the upload.")
    print("Add a Colab secret HF_TOKEN with write access and re-run this cell to publish.")
else:
    api = HfApi(token=hf_token)
    api.create_repo(HF_REPO, repo_type="model", exist_ok=True)

    # 1 · the model, at the repo root (Notebook B loads it from there)
    api.upload_folder(folder_path=f"{ARTIFACTS}/model", repo_id=HF_REPO,
                      allow_patterns=["*.safetensors", "*.json"],
                      commit_message="SENTRY-10: AST fine-tuned on an 11-class ESC-50 subset")

    # 2 · the demo artifacts, staged so the cleaned demo_cache.json is what goes up
    STAGE = "/content/hf_stage/artifacts"
    shutil.rmtree("/content/hf_stage", ignore_errors=True)
    shutil.copytree(f"{ARTIFACTS}/demo_audio", f"{STAGE}/demo_audio")
    if os.path.isdir(f"{ARTIFACTS}/figures"):
        shutil.copytree(f"{ARTIFACTS}/figures", f"{STAGE}/figures")
    with open(f"{ARTIFACTS}/demo_cache.json") as f:
        _cache = json.load(f)
    for _k in ("root", "audiosamples_dir"):       # absolute local / Drive paths
        _cache.pop(_k, None)
    with open(f"{STAGE}/demo_cache.json", "w") as f:
        json.dump(_cache, f, indent=2, default=str)
    api.upload_folder(folder_path=STAGE, path_in_repo="artifacts", repo_id=HF_REPO,
                      commit_message="SENTRY-10 demo artifacts for Notebook B")

    # 3 · a short model card
    card = (
        "# SENTRY-10 acoustic safety classifier\n\n"
        "`MIT/ast-finetuned-audioset-10-10-0.4593` fine-tuned on an 11-class subset of ESC-50 "
        "(10 safety-relevant classes + `background_other`), trained on folds 1-4 and evaluated "
        "on fold 5.\n\n"
        f"Classes: {', '.join(CLASSES)}. Confidence gate tau = {TAU}.\n\n"
        "`artifacts/` holds the demo assets used by the SENTRY-10 live demo notebook.\n\n"
        "**Data credit:** ESC-50 (K. J. Piczak, 2015), licensed CC BY-NC 3.0. The demo clips in "
        "`artifacts/demo_audio/ev_*.wav` are ESC-50 fold-5 clips; non-commercial use only.\n\n"
        "Built as a 4-person team student project; not for safety-critical use.\n")
    api.upload_file(path_or_fileobj=card.encode("utf-8"), path_in_repo="README.md",
                    repo_id=HF_REPO, commit_message="model card")
    print(f"published -> https://huggingface.co/{HF_REPO}")
    for _f in sorted(api.list_repo_files(HF_REPO)):
        print("  ", _f)



---
## Step 9 — handoff check

Run this last. If every line says OK, Notebook B will start cleanly tomorrow. If anything says MISSING,
fix it **now**, not at 9 a.m. on demo day.

In [ ]:

import os, json
checks = []
def chk(label, ok, detail=""):
    checks.append((label, ok, detail))
    print(("OK      " if ok else "MISSING ") + label + ("  " + detail if detail else ""))

chk("model directory", os.path.exists(f"{ARTIFACTS}/model/model.safetensors")
    or os.path.exists(f"{ARTIFACTS}/model/pytorch_model.bin"))
chk("feature extractor config", os.path.exists(f"{ARTIFACTS}/model/preprocessor_config.json"))
chk("demo_cache.json", os.path.exists(f"{ARTIFACTS}/demo_cache.json"))
n_ev = len([f for f in os.listdir(f"{ARTIFACTS}/demo_audio") if f.endswith(".wav")])
chk("8 event clips", n_ev == 8, f"found {n_ev}")
n_pa = len(os.listdir(f"{ARTIFACTS}/demo_audio/pa")) if os.path.exists(f"{ARTIFACTS}/demo_audio/pa") else 0
chk("PA announcements pre-rendered", n_pa >= 5, f"found {n_pa}")
chk("figures", len(os.listdir(f"{ARTIFACTS}/figures")) >= 4)
if demo_cache["metrics"]["have_cross_source"]:
    chk("cross-source results csv", os.path.exists(f"{ARTIFACTS}/cross_source_results.csv"))
else:
    print("SKIPPED  cross-source csv (no AudioSamples) - Steps 7a/7b covered it")

acc_ok = demo_cache["metrics"]["in_domain_accuracy"] > 0.70
chk("in-domain accuracy above 0.70", acc_ok,
    f"{demo_cache['metrics']['in_domain_accuracy']:.3f}")
n_correct = sum(e["model_correct"] for e in cache_events)
chk("model gets at least 6/8 demo events right", n_correct >= 6, f"{n_correct}/8")

print("\n" + ("ALL CLEAR - Notebook B is good to go." if all(c[1] for c in checks)
              else "FIX THE MISSING ITEMS ABOVE BEFORE DEMO DAY."))
print(f"\nEverything Notebook B needs is in:\n  {ARTIFACTS}")


---
## Where the marks are, and which cell earns them

| Rubric criterion | Evidence in this notebook |
|---|---|
| Technical integration | Step 4 (head swap, discriminative LRs), Step 3 (SpecAugment), Step 5 (training curve) |
| Creative integration | Step 8 context layer + LLM fusion, and the PA/TTS callback to Lab B |
| Written report | Steps 6, 7 — both accuracy numbers, the confusion matrices, the τ sweep, the hard-negative table |
| Teamwork & process | The fact that this notebook is separate from the demo notebook, and the Step 9 handoff check, and the Step 8b Hub push that Notebook B downloads |

**Do not delete the cross-source results.** The gap between 0.9-something in-domain and whatever you
get on the course samples is the most interesting thing you will have to say, and examiners reward
teams who volunteer it rather than teams who hide it.